# 15. Samplers and solvers — DPM-Solver++ and complete UniPC predictor-corrector

The sampler is exercised with a learned data-prediction network; there is no analytic oracle or closed-form data stand-in. The **solver equations and predictor/corrector path are kept** while only the sample dimension, network width, batch size, and training budget are reduced.

The UniPC path explicitly performs `UniP -> endpoint model evaluation -> UniC` and retains the multistep history used by the coefficient system.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(5)
device = torch.device("cpu")

## 1. VP schedule and learned data prediction

In [ ]:
def alpha(t):
    return torch.cos(0.5 * math.pi * t)


def sigma(t):
    return torch.sin(0.5 * math.pi * t)


def lambda_t(t):
    return torch.log(alpha(t)) - torch.log(sigma(t))


def inverse_lambda(value):
    return 2.0 / math.pi * torch.atan(torch.exp(-value))


class DataPredictor(nn.Module):
    def __init__(self, data_dim=2, hidden_dim=24):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, x_t, t):
        return self.net(torch.cat([x_t, t[:, None]], dim=-1))


model = DataPredictor().to(device)
clean = torch.randn(16, 2, device=device)
noise = torch.randn_like(clean)
train_t = torch.linspace(0.05, 0.95, 16, device=device)
noisy = (
    alpha(train_t)[:, None] * clean
    + sigma(train_t)[:, None] * noise
)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
for _ in range(5):
    optimizer.zero_grad()
    prediction = model(noisy, train_t)
    loss = F.mse_loss(prediction, clean)
    loss.backward()
    optimizer.step()


def model_data_prediction(x, t):
    if t.ndim == 0:
        t = t.expand(x.size(0))
    return model(x, t)

## 2. DPM-Solver++ first- and second-order data-prediction updates

These updates operate in log-SNR coordinates and use the data prediction `x_0` rather than replacing the solver with an Euler step.

In [ ]:
def dpmpp_first_order(x_s, s, t, model_fn):
    h = lambda_t(t) - lambda_t(s)
    model_s = model_fn(x_s, s)
    phi_1 = torch.expm1(-h)
    return (
        sigma(t) / sigma(s) * x_s
        - alpha(t) * phi_1 * model_s
    )


def dpmpp_second_order(x_s, s, t, model_fn, r1=0.5):
    lambda_s = lambda_t(s)
    h = lambda_t(t) - lambda_s
    s1 = inverse_lambda(lambda_s + r1 * h)

    model_s = model_fn(x_s, s)
    x_s1 = (
        sigma(s1) / sigma(s) * x_s
        - alpha(s1) * torch.expm1(-r1 * h) * model_s
    )
    model_s1 = model_fn(x_s1, s1)

    phi_1 = torch.expm1(-h)
    correction = (
        0.5
        / r1
        * alpha(t)
        * phi_1
        * (model_s1 - model_s)
    )
    return (
        sigma(t) / sigma(s) * x_s
        - alpha(t) * phi_1 * model_s
        - correction
    )


x_s = torch.randn(2, 2, device=device)
s = torch.tensor(0.80, device=device)
t = torch.tensor(0.60, device=device)
first_order = dpmpp_first_order(x_s, s, t, model_data_prediction)
second_order = dpmpp_second_order(x_s, s, t, model_data_prediction)
assert first_order.shape == x_s.shape
assert second_order.shape == x_s.shape

## 3. UniPC B(h) coefficient system

In [ ]:
def unipc_bh_system(
    current_time,
    target_time,
    model_history,
    time_history,
    order,
    sample,
):
    model_s0 = model_history[-1]
    lambda_s0 = lambda_t(current_time)
    lambda_target = lambda_t(target_time)
    h = lambda_target - lambda_s0

    rks = []
    d1_terms = []
    for history_offset in range(1, order):
        previous_time = time_history[-(history_offset + 1)]
        previous_model = model_history[-(history_offset + 1)]
        lambda_previous = lambda_t(previous_time)

        rk = (lambda_previous - lambda_s0) / h
        rks.append(rk)
        d1_terms.append((previous_model - model_s0) / rk)

    rks.append(torch.ones((), device=sample.device))
    rks = torch.stack(rks)

    hh = -h
    h_phi_1 = torch.expm1(hh)
    h_phi_k = h_phi_1 / hh - 1.0
    b_h = torch.expm1(hh)

    factorial = 1.0
    matrix_rows = []
    right_hand_side = []
    for power in range(1, order + 1):
        matrix_rows.append(rks.pow(power - 1))
        right_hand_side.append(h_phi_k * factorial / b_h)

        factorial *= power + 1
        h_phi_k = h_phi_k / hh - 1.0 / factorial

    return {
        "model_s0": model_s0,
        "h_phi_1": h_phi_1,
        "b_h": b_h,
        "matrix": torch.stack(matrix_rows),
        "rhs": torch.stack(right_hand_side),
        "d1_terms": d1_terms,
    }

## 4. UniP predictor and UniC corrector

In [ ]:
def unip_bh_predict(
    sample,
    current_time,
    target_time,
    model_history,
    time_history,
    order,
):
    system = unipc_bh_system(
        current_time,
        target_time,
        model_history,
        time_history,
        order,
        sample,
    )

    if system["d1_terms"]:
        differences = torch.stack(system["d1_terms"], dim=0)
        if order == 2:
            rho = torch.full(
                (1,),
                0.5,
                dtype=sample.dtype,
                device=sample.device,
            )
        else:
            rho = torch.linalg.solve(
                system["matrix"][:-1, :-1],
                system["rhs"][:-1],
            ).to(sample.dtype)
        predictor_residual = torch.einsum(
            "k,kbd->bd",
            rho,
            differences,
        )
    else:
        predictor_residual = torch.zeros_like(sample)

    base = (
        sigma(target_time) / sigma(current_time) * sample
        - alpha(target_time)
        * system["h_phi_1"]
        * system["model_s0"]
    )
    return (
        base
        - alpha(target_time)
        * system["b_h"]
        * predictor_residual
    )


def unic_bh_correct(
    last_sample,
    current_time,
    target_time,
    model_history,
    time_history,
    target_model_output,
    order,
):
    system = unipc_bh_system(
        current_time,
        target_time,
        model_history,
        time_history,
        order,
        last_sample,
    )

    if order == 1:
        rho = torch.full(
            (1,),
            0.5,
            dtype=last_sample.dtype,
            device=last_sample.device,
        )
    else:
        rho = torch.linalg.solve(
            system["matrix"],
            system["rhs"],
        ).to(last_sample.dtype)

    if system["d1_terms"]:
        previous_residual = torch.einsum(
            "k,kbd->bd",
            rho[:-1],
            torch.stack(system["d1_terms"], dim=0),
        )
    else:
        previous_residual = torch.zeros_like(last_sample)

    endpoint_difference = (
        target_model_output - system["model_s0"]
    )
    correction_residual = (
        previous_residual
        + rho[-1] * endpoint_difference
    )

    base = (
        sigma(target_time) / sigma(current_time) * last_sample
        - alpha(target_time)
        * system["h_phi_1"]
        * system["model_s0"]
    )
    return (
        base
        - alpha(target_time)
        * system["b_h"]
        * correction_residual
    )

## 5. Complete multistep UniPC execution

A model-call counter verifies that every transition evaluates the model at the current point and then once more at the freshly predicted endpoint before UniC is applied.

In [ ]:
class CountedModel:
    def __init__(self, model_fn):
        self.model_fn = model_fn
        self.calls = 0

    def __call__(self, x, t):
        self.calls += 1
        return self.model_fn(x, t)


counted_model = CountedModel(model_data_prediction)
sample = torch.randn(2, 2, device=device)
time_values = torch.linspace(0.90, 0.10, 10, device=device)
model_history = []
time_history = []
corrector_count = 0

for step_index in range(len(time_values) - 1):
    current_time = time_values[step_index]
    target_time = time_values[step_index + 1]

    current_model = counted_model(sample, current_time).detach()
    model_history.append(current_model)
    time_history.append(current_time)

    order = min(3, len(model_history))
    predicted_sample = unip_bh_predict(
        sample,
        current_time,
        target_time,
        model_history,
        time_history,
        order,
    )
    endpoint_model = counted_model(
        predicted_sample,
        target_time,
    ).detach()
    sample = unic_bh_correct(
        sample,
        current_time,
        target_time,
        model_history,
        time_history,
        endpoint_model,
        order,
    )
    corrector_count += 1

expected_transitions = len(time_values) - 1
assert counted_model.calls == 2 * expected_transitions
assert corrector_count == expected_transitions
assert torch.isfinite(sample).all()
print("UniPC transitions:", expected_transitions)
print("model evaluations:", counted_model.calls)
print("UniC corrector calls:", corrector_count)

## Structural checklist

- learned data predictor instead of an analytic oracle,
- DPM-Solver++ analytical first/second-order updates,
- UniPC log-SNR history and B(h) coefficient system,
- UniP predictor,
- fresh endpoint model evaluation,
- UniC corrector on every transition,
- multistep warm-up to order 3.

Only execution-scale tensors and the predictor training budget are reduced.